In [18]:
from pathlib import Path
import pandas as pd

from trino_stack.lakehouse import Lakehouse


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

WORKLOAD_ROOT = Path("/mnt/primary/Main/Workloads")

WORKLOAD_NAMES = [
    "tpcds",
    "gpt_generated_tpcds",
]

CATALOG = "iceberg"
SCHEMA = "tpcds"


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def extract_overview_from_output(out):
    """
    Extract the raw overview dict from lh.workload_diversity_metrics(...).

    This is intentionally defensive, in case the wrapper returns slightly
    different structures.
    """
    if isinstance(out, dict):
        if "report" in out and isinstance(out["report"], dict):
            if "overview" in out["report"]:
                return out["report"]["overview"]

        if "overview" in out:
            return out["overview"]

    raise ValueError(
        "Could not find an overview dict in the workload_diversity_metrics output."
    )


def resolve_workload_path(workload_name_or_path, workload_root=WORKLOAD_ROOT):
    """
    Accept either:
      - workload directory name, e.g. 'tpcds'
      - absolute path, e.g. '/mnt/primary/Main/Workloads/tpcds'
    """
    path = Path(workload_name_or_path)

    if path.is_absolute():
        return path

    return workload_root / path


def analyse_workload_set(
    *,
    lh,
    workload_names,
    workload_root=WORKLOAD_ROOT,
    catalog=CATALOG,
    schema=SCHEMA,
):
    """
    Generate one overview-metric table for multiple workloads.

    Each row is one workload.
    Each metric from report['overview'] becomes one column.
    """
    rows = []
    failures = []

    for workload_name in workload_names:
        workload_path = resolve_workload_path(
            workload_name,
            workload_root=workload_root,
        )

        print(f"\n=== Analysing workload: {workload_path.name} ===")

        try:
            plan_bundle = lh.load_query_plans(
                workload_path=str(workload_path),
                schema=schema,
            )

            out = lh.workload_diversity_metrics(
                plan_bundle,
                catalog=catalog,
                schema=schema,
            )

            overview = extract_overview_from_output(out)

            row = {
                "workload": plan_bundle.get("workload_name", workload_path.name),
                "workload_path": str(workload_path),
                "query_count": plan_bundle.get("query_count"),
                "ok": plan_bundle.get("ok"),
                "failed": plan_bundle.get("failed"),
            }

            row.update(overview)
            rows.append(row)

        except Exception as e:
            failures.append({
                "workload": workload_path.name,
                "workload_path": str(workload_path),
                "error": repr(e),
            })

            print(f"FAILED: {workload_path.name}")
            print(repr(e))

    metrics_df = pd.DataFrame(rows)

    if not metrics_df.empty:
        front_cols = [
            "workload",
            "workload_path",
            "query_count",
            "ok",
            "failed",
        ]

        metric_cols = [
            c for c in metrics_df.columns
            if c not in front_cols
        ]

        metrics_df = metrics_df[front_cols + metric_cols]

    failures_df = pd.DataFrame(failures)

    return metrics_df, failures_df

lh = Lakehouse.from_release(
    instance_name="lakehouse-a",
    namespace="pgr24james",
    sync_schemas=True,
    verbose=False,
)

overview_df, failures_df = analyse_workload_set(
    lh=lh,
    workload_names=WORKLOAD_NAMES,
    catalog="iceberg",
    schema="tpcds",
)

df = overview_df.copy()

df["group"] = "Other"
df.loc[df["workload"] == "tpcds", "group"] = "TPC-DS"
df.loc[df["workload"].str.contains("_low_"), "group"] = "Low reasoning"
df.loc[df["workload"].str.contains("_medium_"), "group"] = "Medium reasoning"
df.loc[df["workload"].str.contains("_high_"), "group"] = "High reasoning"

summary = (
    df[df["group"].isin(["TPC-DS", "Low reasoning", "Medium reasoning", "High reasoning"])]
    .drop(columns=["workload", "workload_path"], errors="ignore")
    .groupby("group", as_index=False)
    .mean(numeric_only=True)
)

summary
summary.to_csv(
    "/mnt/primary/Main/Workloads/reasoning_model_diversity_summary.csv",
    index=False,
)

Checking Trino health ...

=== Analysing workload: tpcds ===
workload  sql_queries  planned_queries  table_coverage_pct  column_coverage_pct  join_edge_coverage_pct  table_usage_entropy  column_usage_entropy  unique_table_set_ratio  unique_plan_graphs  plan_uniqueness_ratio  operator_types_observed  operator_type_entropy  unique_operator_instances  operator_ngram_entropy  mean_nn_plan_distance  min_nn_plan_distance  plan_graph_vendi_score
   tpcds           99               99               100.0                58.12                   68.29               0.8447                0.8809                  0.6566                  84                 0.8485                       29                 0.6801                       2740                  0.7257                 0.2437                   0.0                 22.2601

=== Analysing workload: gpt_generated_tpcds ===
           workload  sql_queries  planned_queries  table_coverage_pct  column_coverage_pct  join_edge_coverage_pct  table_usag

In [17]:
from pathlib import Path
from collections import Counter
import pandas as pd

from trino_stack.lakehouse import Lakehouse
from loader.workload_analysis import (
    dags_from_plan_bundle,
    operator_type_counts_many,
)

# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

WORKLOAD_ROOT = Path("/mnt/primary/Main/Workloads")
SCHEMA = "tpcds"

WORKLOAD_NAMES = [
    "tpcds",

    "gpt_generated_grid_low_w2",
    "gpt_generated_grid_low_w4",
    "gpt_generated_grid_low_w8",
    "gpt_generated_grid_low_w16",
    "gpt_generated_grid_low_w32",
    "gpt_generated_grid_low_w64",
    "gpt_generated_grid_low_w128",

    "gpt_generated_grid_medium_w2",
    "gpt_generated_grid_medium_w4",
    "gpt_generated_grid_medium_w8",
    "gpt_generated_grid_medium_w16",
    "gpt_generated_grid_medium_w32",
    "gpt_generated_grid_medium_w64",
    "gpt_generated_grid_medium_w128",

    "gpt_generated_grid_high_w2",
    "gpt_generated_grid_high_w4",
    "gpt_generated_grid_high_w8",
    "gpt_generated_grid_high_w16",
    "gpt_generated_grid_high_w32",
    "gpt_generated_grid_high_w64",
    "gpt_generated_grid_high_w128",
]


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def reasoning_group(workload_name: str) -> str:
    workload_name = str(workload_name)

    if workload_name == "tpcds":
        return "tpcds"

    if "_low_" in workload_name:
        return "low"

    if "_medium_" in workload_name:
        return "medium"

    if "_high_" in workload_name:
        return "high"

    return "other"


# ---------------------------------------------------------------------
# Load lakehouse
# ---------------------------------------------------------------------

lh = Lakehouse.from_release(
    instance_name="lakehouse-a",
    namespace="pgr24james",
    sync_schemas=True,
    verbose=False,
)


# ---------------------------------------------------------------------
# Collect operator counts per workload and per reasoning group
# ---------------------------------------------------------------------

operator_counts_by_workload = {}
operator_counts_by_group = {
    "tpcds": Counter(),
    "low": Counter(),
    "medium": Counter(),
    "high": Counter(),
}

failures = []

for workload_name in WORKLOAD_NAMES:
    workload_path = WORKLOAD_ROOT / workload_name
    group = reasoning_group(workload_name)

    print(f"Analysing {workload_name}...")

    try:
        plan_bundle = lh.load_query_plans(
            workload_path=str(workload_path),
            schema=SCHEMA,
        )

        dags_by_query = dags_from_plan_bundle(plan_bundle)
        counts = operator_type_counts_many(dags_by_query.values())

        operator_counts_by_workload[workload_name] = counts

        if group in operator_counts_by_group:
            operator_counts_by_group[group].update(counts)

    except Exception as e:
        failures.append({
            "workload": workload_name,
            "workload_path": str(workload_path),
            "error": repr(e),
        })
        print(f"FAILED: {workload_name}: {repr(e)}")


# ---------------------------------------------------------------------
# Build comparison table
# ---------------------------------------------------------------------

all_operator_types = sorted(set().union(*[
    set(counter.keys())
    for counter in operator_counts_by_group.values()
]))

operator_group_df = pd.DataFrame([
    {
        "operator_type": op,
        "tpcds_count": operator_counts_by_group["tpcds"].get(op, 0),
        "low_count": operator_counts_by_group["low"].get(op, 0),
        "medium_count": operator_counts_by_group["medium"].get(op, 0),
        "high_count": operator_counts_by_group["high"].get(op, 0),
    }
    for op in all_operator_types
])

operator_group_df["present_in_tpcds"] = operator_group_df["tpcds_count"] > 0
operator_group_df["present_in_low"] = operator_group_df["low_count"] > 0
operator_group_df["present_in_medium"] = operator_group_df["medium_count"] > 0
operator_group_df["present_in_high"] = operator_group_df["high_count"] > 0
operator_group_df["present_in_any_generated"] = (
    operator_group_df["present_in_low"]
    | operator_group_df["present_in_medium"]
    | operator_group_df["present_in_high"]
)


# ---------------------------------------------------------------------
# Operators present in TPC-DS but absent from all generated workloads
# ---------------------------------------------------------------------

tpcds_only_df = (
    operator_group_df[
        operator_group_df["present_in_tpcds"]
        & ~operator_group_df["present_in_any_generated"]
    ]
    .sort_values("tpcds_count", ascending=False)
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Operators present in TPC-DS but missing by reasoning level
# ---------------------------------------------------------------------

tpcds_missing_by_reasoning_df = (
    operator_group_df[operator_group_df["present_in_tpcds"]]
    .copy()
)

tpcds_missing_by_reasoning_df["missing_from_low"] = ~tpcds_missing_by_reasoning_df["present_in_low"]
tpcds_missing_by_reasoning_df["missing_from_medium"] = ~tpcds_missing_by_reasoning_df["present_in_medium"]
tpcds_missing_by_reasoning_df["missing_from_high"] = ~tpcds_missing_by_reasoning_df["present_in_high"]

tpcds_missing_by_reasoning_df = (
    tpcds_missing_by_reasoning_df
    .sort_values("tpcds_count", ascending=False)
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Export results
# ---------------------------------------------------------------------

operator_group_df.to_csv(
    WORKLOAD_ROOT / "operator_type_counts_by_reasoning.csv",
    index=False,
)

tpcds_only_df.to_csv(
    WORKLOAD_ROOT / "tpcds_only_operator_types.csv",
    index=False,
)

tpcds_missing_by_reasoning_df.to_csv(
    WORKLOAD_ROOT / "tpcds_operator_missing_by_reasoning.csv",
    index=False,
)

failures_df = pd.DataFrame(failures)

if not failures_df.empty:
    failures_df.to_csv(
        WORKLOAD_ROOT / "operator_type_analysis_failures.csv",
        index=False,
    )


# ---------------------------------------------------------------------
# Display key outputs
# ---------------------------------------------------------------------

print("\n=== Operators present in TPC-DS but absent from all generated workloads ===")
display(tpcds_only_df)

print("\n=== TPC-DS operators and whether each reasoning level misses them ===")
display(tpcds_missing_by_reasoning_df)

print("\n=== Full operator count table by group ===")
display(operator_group_df)

if not failures_df.empty:
    print("\n=== Failures ===")
    display(failures_df)

Checking Trino health ...
Analysing tpcds...
Analysing gpt_generated_grid_low_w2...
Analysing gpt_generated_grid_low_w4...
Analysing gpt_generated_grid_low_w8...
Analysing gpt_generated_grid_low_w16...
Analysing gpt_generated_grid_low_w32...
Analysing gpt_generated_grid_low_w64...
Analysing gpt_generated_grid_low_w128...
Analysing gpt_generated_grid_medium_w2...
Analysing gpt_generated_grid_medium_w4...
Analysing gpt_generated_grid_medium_w8...
Analysing gpt_generated_grid_medium_w16...
Analysing gpt_generated_grid_medium_w32...
Analysing gpt_generated_grid_medium_w64...
Analysing gpt_generated_grid_medium_w128...
Analysing gpt_generated_grid_high_w2...
Analysing gpt_generated_grid_high_w4...
Analysing gpt_generated_grid_high_w8...
Analysing gpt_generated_grid_high_w16...
Analysing gpt_generated_grid_high_w32...
Analysing gpt_generated_grid_high_w64...
Analysing gpt_generated_grid_high_w128...

=== Operators present in TPC-DS but absent from all generated workloads ===


,operator_type,tpcds_count,low_count,medium_count,high_count,present_in_tpcds,present_in_low,present_in_medium,present_in_high,present_in_any_generated
0,EnforceSingleRow,8,0,0,0,True,False,False,False,False
1,Values,6,0,0,0,True,False,False,False,False
2,AssignUniqueId,3,0,0,0,True,False,False,False,False
3,Limit,2,0,0,0,True,False,False,False,False
4,SemiJoin,1,0,0,0,True,False,False,False,False



=== TPC-DS operators and whether each reasoning level misses them ===


,operator_type,tpcds_count,low_count,medium_count,high_count,present_in_tpcds,present_in_low,present_in_medium,present_in_high,present_in_any_generated,missing_from_low,missing_from_medium,missing_from_high
0,LocalExchange,1237,5294,5833,7419,True,True,True,True,True,False,False,False
1,RemoteSource,1195,5790,5772,7791,True,True,True,True,True,False,False,False
2,InnerJoin,653,2851,2665,3923,True,True,True,True,True,False,False,False
3,Aggregate,626,2719,3010,3911,True,True,True,True,True,False,False,False
4,ScanFilter,395,1837,1937,2811,True,True,True,True,True,False,False,False
5,Project,324,1490,1466,2453,True,True,True,True,True,False,False,False
6,ScanFilterProject,322,210,287,617,True,True,True,True,True,False,False,False
7,TopNPartial,218,697,1259,911,True,True,True,True,True,False,False,False
8,TableScan,155,2384,2203,2678,True,True,True,True,True,False,False,False
9,Output,99,700,700,700,True,True,True,True,True,False,False,False



=== Full operator count table by group ===


,operator_type,tpcds_count,low_count,medium_count,high_count,present_in_tpcds,present_in_low,present_in_medium,present_in_high,present_in_any_generated
0,Aggregate,626,2719,3010,3911,True,True,True,True,True
1,AssignUniqueId,3,0,0,0,True,False,False,False,False
2,CrossJoin,46,7,3,1,True,True,True,True,True
3,EnforceSingleRow,8,0,0,0,True,False,False,False,False
4,Filter,29,7,17,21,True,True,True,True,True
5,FilterProject,42,8,15,31,True,True,True,True,True
6,FullJoin,2,161,215,249,True,True,True,True,True
7,GroupId,20,77,71,107,True,True,True,True,True
8,InnerJoin,653,2851,2665,3923,True,True,True,True,True
9,LeftJoin,31,420,485,656,True,True,True,True,True
